# Stopping strategies across learning

Three panels, after Pettit et al.'s strategy analysis, adapted to this task.

The task differs from theirs in a way that shapes everything here: there is **one** reward
zone, not two, and trials differ instead by whether that zone is **visually cued**. So the
strategy ladder is not *which* reward an animal goes for, it is **whether the animal can
find the zone without being shown it** — which is exactly the path-integration question the
rest of the paper is about.

Strategies were derived from the data rather than assumed. Two quantities separate them:

- **stop selectivity** — the concentration of stops inside the reward zone, expressed
  relative to chance (1.0 = stops spread uniformly along the track);
- **uncued hit rate** — whether the animal actually solves trials with no cue.

| strategy | what it looks like | n sessions |
|---|---|---|
| **non-selective** | stops constantly and everywhere (median 20.7 stops/trial), selectivity ≈ chance | 3 |
| **cue-dependent** | sharply selective when cued (5.4×) but not when uncued (2.0×); uncued hit rate 0.11 | 27 |
| **partial** | selective on both (3.6× / 3.2×), uncued hit rate 0.55 | 22 |
| **expert** | selective on both (4.7× / 4.1×), uncued hit rate 0.86 | 27 |

The gap between cue-dependent and expert is the informative one: both stop precisely at the
reward zone when it is marked, and only one can do it when it is not.

In [ ]:
import sys, os, glob, re, json, warnings
sys.path.insert(0, '/Users/harryclark/Documents/spatial-manifolds/src')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pynapple as nap
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Arial'

# reuse the clipped stop extraction from the raster notebook -- one definition only
_NB = ('/Users/harryclark/Documents/spatial-manifolds/scripts/figures/'
       'AnchorDynamics2026/lick_raster_by_trial_type.ipynb')
_g = {}
for _c in json.load(open(_NB))['cells']:
    if _c['cell_type'] != 'code':
        continue
    _s = ''.join(_c['source'])
    if _s.startswith('MOUSE, DAY') or 'INV = pd.DataFrame' in _s:
        continue
    exec(compile(_s, '<raster-nb>', 'exec'), _g)
globals().update({k: v for k, v in _g.items() if not k.startswith('__')})

STRAT_ORDER = ['cue-dependent', 'partial', 'expert']
STRAT_COL = dict(zip(STRAT_ORDER, ['#94b8d8', '#4a7fb5', '#1f3f73']))
OVERSTOP = 15.0     # stops/trial above which a session is flagged as over-stopping
NBINS = 20
EDGES = np.linspace(0, TL, NBINS + 1)
RZ_MASK = (EDGES[:-1] >= RZ[0]) & (EDGES[1:] <= RZ[1])
print('setup ready')

## Per-session stopping profiles and strategy assignment

In [ ]:
BOX = 30.0     # start black box; a stop inside it is not a choice about the zone

def session_profile(mouse, day):
    """Stop-position profiles and outcome, per trial type.

    The profile is normalised WITHIN TRIAL -- each trial's stops sum to 1 before
    averaging -- so a trial with twenty stops counts the same as a trial with one.
    Pooling raw stops instead lets the heaviest stoppers dominate the histogram and
    manufactures a reward-zone peak out of dwell time rather than choice.

    The first stop of each trial is kept as a separate measure. It is diagnostic for
    animals that stop constantly, but it is NOT a good general measure here: expert
    animals stop several times per trial and their first stop is usually early on the
    track, so first-stop selectivity collapses to near chance (1.84 vs 5.27 for the
    within-trial normalised measure) even though their stopping is highly selective.
    """
    beh, trials, stops = load_vr(mouse, day)
    if len(stops) < 50:
        return None
    stops = stops[stops.pos > BOX]
    out = dict(mouse=mouse, day=day, n_trials=len(trials), n_stops=len(stops))
    prof = {}
    for tt, name in (('b', 'cued'), ('nb', 'uncued')):
        keep = set(trials[trials.type == tt].number.astype(int))
        S = stops[stops.trial.isin(keep)]
        if len(keep) < 10 or len(S) < 20:
            return None
        acc, ntr = np.zeros(NBINS), 0
        for _t, grp in S.groupby('trial'):
            h, _ = np.histogram(grp.pos, bins=EDGES)
            if h.sum():
                acc += h / h.sum(); ntr += 1
        p = acc / max(ntr, 1)
        prof[name] = p
        # selectivity = (share of stopping inside the zone) / (share of track the
        # zone occupies); 1.0 means stopping spread uniformly along the track
        out[f'sel_{name}'] = float(p[RZ_MASK].sum() / RZ_MASK.mean())
        F = S.sort_values(['trial', 't']).groupby('trial', as_index=False).first()
        hf, _ = np.histogram(F.pos, bins=EDGES); pf = hf / max(hf.sum(), 1)
        prof[f'{name}_first'] = pf
        out[f'sel_first_{name}'] = float(pf[RZ_MASK].sum() / RZ_MASK.mean())
        out[f'stops_per_trial_{name}'] = len(S) / len(keep)
        out[f'hit_{name}'] = float((trials[trials.type == tt].performance == 'hit').mean())
    return out, prof, stops, trials

def classify_strategy(r):
    """Ladder derived from the observed distributions (see the markdown above)."""
    if r['sel_cued'] < 2.0 and r['sel_uncued'] < 2.0:
        return 'non-selective'          # stops everywhere; hits the zone by accident
    if r['sel_uncued'] < 2.0 or r['hit_uncued'] < 0.25:
        return 'cue-dependent'          # precise only when shown the zone
    if r['hit_uncued'] >= 0.70:
        return 'expert'
    return 'partial'

rows, profiles = [], {}
for d in sorted(glob.glob(f'{SOURCE_PATH}M*/D*/VR')):
    m = re.search(r'M(\d+)/D(\d+)/VR', d)
    mo, dy = int(m.group(1)), int(m.group(2))
    bp, cp = vr_paths(mo, dy)
    if not (os.path.exists(bp) and os.path.exists(cp)):
        continue
    try:
        res = session_profile(mo, dy)
    except Exception:
        continue
    if res is None:
        continue
    rec, prof, _, _ = res
    rec['strategy'] = classify_strategy(rec)
    rec['over_stopping'] = rec['stops_per_trial_cued'] > OVERSTOP
    rows.append(rec); profiles[(mo, dy)] = prof
B = pd.DataFrame(rows).sort_values(['mouse', 'day'])
B['sess'] = B.groupby('mouse').cumcount() + 1
print(f'{len(B)} sessions classified\n')
print(B.strategy.value_counts().reindex(STRAT_ORDER).to_string())
print(f"\nover-stopping (>{OVERSTOP:.0f} stops/trial): {int(B.over_stopping.sum())} sessions "
      f"-- a separate axis, not a strategy: {B[B.over_stopping].strategy.value_counts().to_dict()}")
print()
print(B.groupby('strategy')[['sel_cued','sel_uncued','hit_cued','hit_uncued',
                             'stops_per_trial_cued']].median().round(2).to_string())

## Panel d — what each strategy looks like

Mean stop-position profile for each strategy, cued and uncued. This is the panel that
justifies the categories: if the four rows do not look different, the classification is
arbitrary.

In [ ]:
fig, axes = plt.subplots(len(STRAT_ORDER), 1, figsize=(4.2, 5.4), sharex=True)
for ax, strat in zip(axes, STRAT_ORDER):
    keys = [(r.mouse, r.day) for r in B[B.strategy == strat].itertuples()]
    if not keys:
        ax.axis('off'); continue
    x = EDGES[:-1] + np.diff(EDGES) / 2
    for name, col, ls in (('cued', CUED_C, '-'), ('uncued', UNCUED_C, '-')):
        P = np.stack([profiles[k][name] for k in keys])
        m_, se = P.mean(0), P.std(0) / np.sqrt(len(P))
        ax.plot(x, m_, color=col, lw=1.5, ls=ls, label=name)
        ax.fill_between(x, m_-se, m_+se, color=col, alpha=.22, linewidth=0, edgecolor='none')
    ax.axvspan(*RZ, color='#9ecae1', alpha=.35, lw=0, zorder=0)
    ax.set_ylabel('P(stop)', fontsize=8)
    ax.text(.02, .86, f'{strat}  (n={len(keys)})', transform=ax.transAxes,
            fontsize=8.5, color=STRAT_COL[strat], weight='bold')
    ax.spines[['top','right']].set_visible(False)
axes[0].legend(fontsize=7, frameon=False, loc='upper right')
axes[-1].set_xlabel('Position (cm)')
plt.tight_layout()
plt.savefig(f'{FIGPATH}/strategy_profiles.pdf', dpi=200, bbox_inches='tight')
plt.show()

## Panel e — coefficient of partial determination

How much of the trial-by-trial stopping pattern does each strategy account for, and how
does that change across sessions?

Stop counts are modelled per (trial × position bin) with three regressors:

- **uniform** — stopping unrelated to position
- **cued zone** — the reward zone, on cued trials only
- **uncued zone** — the reward zone, on uncued trials

CPD for a regressor is the proportional increase in residual error when it is dropped,
so it measures unique explanatory contribution. The cued and uncued regressors separate
"solving by looking" from "solving by path integration" — which a single reward-zone
regressor would merge.

In [ ]:
def cpd(mouse, day):
    beh, trials, stops = load_vr(mouse, day)
    tnums = trials.number.astype(int).values
    is_cued = (trials.type == 'b').values
    idx = {t: i for i, t in enumerate(tnums)}
    Y = np.zeros((len(tnums), NBINS))
    for r in stops.itertuples():
        if r.trial in idx:
            Y[idx[r.trial], min(int(r.pos / TL * NBINS), NBINS - 1)] += 1
    y = Y.ravel()
    rz = np.tile(RZ_MASK.astype(float), len(tnums))
    cue = np.repeat(is_cued.astype(float), NBINS)
    X = np.column_stack([np.ones_like(y), rz * cue, rz * (1 - cue)])
    names = ['uniform', 'cued zone', 'uncued zone']
    def sse(Xs):
        b, *_ = np.linalg.lstsq(Xs, y, rcond=None)
        return float(((y - Xs @ b) ** 2).sum())
    full = sse(X)
    out = {}
    for j, nm in enumerate(names):
        if j == 0:
            continue
        red = sse(np.delete(X, j, axis=1))
        out[nm] = 100 * (red - full) / red if red > 0 else np.nan
    return out

cpd_rows = []
for r in B.itertuples():
    try:
        c = cpd(r.mouse, r.day)
    except Exception:
        continue
    cpd_rows.append(dict(mouse=r.mouse, day=r.day, sess=r.sess, strategy=r.strategy, **c))
C = pd.DataFrame(cpd_rows)
print(C.groupby('strategy')[['cued zone','uncued zone']].median().round(2).to_string())

fig, ax = plt.subplots(figsize=(5.2, 2.6))
M = C.groupby('sess')[['cued zone','uncued zone']].mean().T
im = ax.imshow(M.values, aspect='auto', cmap='inferno', vmin=0,
               extent=[.5, M.shape[1]+.5, 1.5, -.5])
ax.set_yticks([0,1]); ax.set_yticklabels(M.index, fontsize=9)
ax.set_xlabel('Session'); ax.set_title('CPD (%)', fontsize=9)
cb = plt.colorbar(im, ax=ax, fraction=.05, pad=.02); cb.outline.set_visible(False)
for sp in ax.spines.values(): sp.set_visible(False)
plt.tight_layout(); plt.savefig(f'{FIGPATH}/strategy_cpd.pdf', dpi=200, bbox_inches='tight')
plt.show()

## Panel f — strategy by animal and session

In [ ]:
rank = (B.assign(k=B.strategy.map({s:i for i,s in enumerate(STRAT_ORDER)}))
         .groupby('mouse').k.mean().sort_values(ascending=False).index.tolist())
G = np.full((len(rank), int(B.sess.max())), np.nan)
for i, mo in enumerate(rank):
    g = B[B.mouse == mo]
    G[i, g.sess.values-1] = g.strategy.map({s:i_ for i_,s in enumerate(STRAT_ORDER)}).values
from matplotlib.colors import ListedColormap, BoundaryNorm
cmap = ListedColormap([STRAT_COL[s] for s in STRAT_ORDER])
norm = BoundaryNorm(np.arange(-.5, len(STRAT_ORDER)), cmap.N)
fig, ax = plt.subplots(figsize=(5.6, 3.0))
ax.imshow(G, aspect='auto', cmap=cmap, norm=norm, interpolation='nearest',
          extent=[.5, G.shape[1]+.5, len(rank)+.5, .5])
ax.set_yticks(range(1, len(rank)+1)); ax.set_yticklabels([f'M{m}' for m in rank], fontsize=8)
ax.set_xlabel('Recording session'); ax.set_ylabel('Animal (ranked)')
# NOT a learning axis: animals were trained to expert before implantation, so this
# is recording order. Strategy is a stable property of the animal here, not a stage
# it passes through -- M20 is cue-dependent in all 13 of its sessions.
for sp in ax.spines.values(): sp.set_visible(False)
handles = [plt.Rectangle((0,0),1,1,color=STRAT_COL[s]) for s in STRAT_ORDER]
ax.legend(handles, STRAT_ORDER, fontsize=7, frameon=False,
          loc='center left', bbox_to_anchor=(1.02, .5))
plt.tight_layout(); plt.savefig(f'{FIGPATH}/strategy_by_session.pdf', dpi=200, bbox_inches='tight')
plt.show()
print(pd.crosstab(B.mouse, B.strategy).reindex(columns=STRAT_ORDER, fill_value=0).to_string())